# Calculate Performance Metrics (Single Model, Plain Python)

## Set Up

### Load packages

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn import metrics

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", 200)

### Helper functions

In [2]:
def load_dataframe(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported input file type: {suffix}")


def normalize_columns(columns):
    if columns is None:
        return []
    if isinstance(columns, str):
        return [col.strip() for col in columns.split(",") if col.strip()]
    return list(columns)


def require_columns(df, columns):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")


def check_unique_keys(df, id_cols, label):
    duplicated_count = int(df.duplicated(id_cols).sum())
    if duplicated_count:
        raise ValueError(
            f"{label} has {duplicated_count:,} duplicate rows for "
            f"join keys {id_cols}. Update id_cols before joining."
        )


def load_model_output(
    path,
    id_cols,
    target_col,
    detected_entities_col,
    predicted_flag_col,
):
    model_df = load_dataframe(path)
    require_columns(model_df, id_cols + [target_col, detected_entities_col])
    check_unique_keys(model_df, id_cols, f"Model output {path}")

    model_df = model_df[id_cols + [target_col, detected_entities_col]].copy()
    detected_entities = (
        model_df[detected_entities_col].fillna("").astype(str).str.strip()
    )
    no_detection_values = {"", "[]", "nan", "None"}
    model_df[predicted_flag_col] = (
        ~detected_entities.isin(no_detection_values)
    ).astype(int)

    return model_df.drop(columns=[detected_entities_col])


def load_manual_annotations(path, id_cols, manual_flag_col):
    manual_df = load_dataframe(path)
    require_columns(manual_df, id_cols + [manual_flag_col])
    check_unique_keys(manual_df, id_cols, f"Manual annotations {path}")

    manual_df = manual_df[id_cols + [manual_flag_col]].copy()
    manual_df[manual_flag_col] = manual_df[manual_flag_col].astype(int)
    return manual_df


def join_model_and_manual_dfs(
    model_output_df,
    manual_annotated_df,
    id_cols,
):
    return model_output_df.merge(
        manual_annotated_df,
        on=id_cols,
        how="inner",
        validate="one_to_one",
    )


def compute_and_plot_cm(
    combined_df,
    manual_flag_col,
    predicted_flag_col,
    title=None,
):
    actual = combined_df[manual_flag_col].astype(int)
    predicted = combined_df[predicted_flag_col].astype(int)

    cm = metrics.confusion_matrix(actual, predicted, labels=[0, 1])
    cm_display = metrics.ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[0, 1],
    )
    cm_display.plot()
    if title:
        plt.title(title)
    plt.show()


def inspect_missed_pii(
    combined_df,
    id_cols,
    target_col,
    manual_flag_col,
    predicted_flag_col,
):
    missed_pii = combined_df[
        combined_df[manual_flag_col] > combined_df[predicted_flag_col]
    ]
    return missed_pii[
        id_cols + [target_col, manual_flag_col, predicted_flag_col]
    ]


def return_metrics(
    combined_df,
    id_cols,
    manual_flag_col,
    predicted_flag_col,
):
    ordered_df = combined_df.sort_values(id_cols)
    actual = ordered_df[manual_flag_col].astype(int)
    predicted = ordered_df[predicted_flag_col].astype(int)

    return pd.Series(
        {
            "recall": metrics.recall_score(actual, predicted, zero_division=0),
            "precision": metrics.precision_score(
                actual, predicted, zero_division=0
            ),
            "accuracy": metrics.accuracy_score(actual, predicted),
            "f1": metrics.f1_score(actual, predicted, zero_division=0),
        }
    )

### Set parameters

In [3]:
# Replace these paths and column names for your local data.
evaluation_sets = {
    "all_eval": {
        "model_output_path": Path(
            "data/model_outputs/default_spacy_ner_results.csv"
        ),
        "manual_annotated_path": Path("data/model_inputs/all_eval_data.csv"),
    },
}

# For additional evaluation subsets, add more entries to evaluation_sets.
# Example:
# evaluation_sets["longest"] = {
#     "model_output_path": Path("data/model_outputs/model_longest.csv"),
#     "manual_annotated_path": Path("data/longest_unique_start_records.csv"),
# }

target_col = "text"
id_cols = ["encounter_id", "lab_eventnumber"]
manual_flag_col = "manual_pii_flag"
detected_entities_col = "detected_entities"
predicted_flag_col = "predicted_pii_flag"

id_col_list = normalize_columns(id_cols)

## Load Model Outputs and Annotated Ground Truth Data

In [ ]:
loaded_sets = {}

for name, paths in evaluation_sets.items():
    model_output_df = load_model_output(
        paths["model_output_path"],
        id_cols=id_col_list,
        target_col=target_col,
        detected_entities_col=detected_entities_col,
        predicted_flag_col=predicted_flag_col,
    )
    manual_annotated_df = load_manual_annotations(
        paths["manual_annotated_path"],
        id_cols=id_col_list,
        manual_flag_col=manual_flag_col,
    )

    loaded_sets[name] = {
        "model_output": model_output_df,
        "manual_annotated": manual_annotated_df,
    }

    print(
        f"{name}: loaded {len(model_output_df):,} model rows "
        f"and {len(manual_annotated_df):,} annotated rows"
    )

## Join and Compare Model and Manual Results

In [ ]:
compare_dfs = {}

for name, dfs in loaded_sets.items():
    compare_df = join_model_and_manual_dfs(
        dfs["model_output"],
        dfs["manual_annotated"],
        id_cols=id_col_list,
    )
    compare_dfs[name] = compare_df
    print(f"{name}: joined {len(compare_df):,} rows")
    display(compare_df.head())

In [ ]:
for name, compare_df in compare_dfs.items():
    compute_and_plot_cm(
        compare_df,
        manual_flag_col=manual_flag_col,
        predicted_flag_col=predicted_flag_col,
        title=f"Confusion Matrix: {name}",
    )

## Inspect PII Missed by Model

In [ ]:
missed_pii_dfs = {}

for name, compare_df in compare_dfs.items():
    missed_pii_df = inspect_missed_pii(
        compare_df,
        id_cols=id_col_list,
        target_col=target_col,
        manual_flag_col=manual_flag_col,
        predicted_flag_col=predicted_flag_col,
    )
    missed_pii_dfs[name] = missed_pii_df
    print(f"{name}: {len(missed_pii_df):,} missed PII rows")
    display(missed_pii_df)

## F1, Precision, Accuracy, Recall

Definitions:

- F1 = harmonic mean of precision and recall = 2 * P * R / (P + R)
- Precision = how many of the positive predictions are actually correct = TP / (TP + FP)
- Accuracy = proportion of correct predictions out of all predictions = (TP + TN) / Total Records
- Recall = how many of the actual positive cases were correctly identified = TP / (TP + FN)

Notes:

For PII detection, **recall** is often the most important. We want to capture all potential positives for human review; we do not need to be highly targeted and only flag true positives.

When there is low prevalence of true PII, **precision** is expected to be low since there are very few positive cases. For example, if there is only 1 case of PII and the model flags that 1 case plus 1 other case, precision is 0.5. Low precision also lowers F1.

In [ ]:
metrics_summary_df = pd.DataFrame(
    {
        name: return_metrics(
            compare_df,
            id_cols=id_col_list,
            manual_flag_col=manual_flag_col,
            predicted_flag_col=predicted_flag_col,
        )
        for name, compare_df in compare_dfs.items()
    }
).T

display(metrics_summary_df)